# MLP diagnosis and regularisation (Task 2)

Task 1 gave a test accuracy of 0.356 for the tuned ReLU network, with a large train/val gap and an explicit
open question in its Conclusion: *`fit_with_best` picked the min-val_loss epoch (8), but val_acc peaks at
epoch 12 - is val_loss the right selection criterion?*

This notebook answers three questions:

1. **Does regularisation help?** Compare baseline / dropout / weight decay / early stopping on the same
   data, model, and number of epochs.
2. **Is the learning rate well chosen?** Task 1 Part A oscillated with SGD lr=0.2; measure the effect of
   three learning rates.
3. **Which epoch should we keep?** Compare test accuracy when we select by *last epoch*, *min val_loss*,
   or *max val_acc*.

References

- TP06 Part B (regularisation: weight decay, dropout, early stopping).
- TP06 Part E "Training doctor" Patient 1 (learning rate too large).
- TP04 Exercise 20 / 24 (evaluate, train_one_epoch, fit) — reused through `museum_guide.train`.

All figures and metrics saved under `results/`.


## Setup

In [ ]:
import os
import sys
import copy
import json
from pathlib import Path

_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root / "src"))
print("cwd:", os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from museum_guide.data import get_dataloaders
from museum_guide.mlp import make_tuned_mlp, make_mlp, init_weights
from museum_guide.train import fit_regularized, EarlyStopping, evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(2)
print("device:", device)

In [ ]:
# Same data loaders as Task 1 (batch_size=128, same splits).
BATCH_SIZE = 128
train_loader, val_loader, test_loader = get_dataloaders("mlp", batch_size=BATCH_SIZE)
loss_fn = nn.CrossEntropyLoss()

N_EPOCHS = 20            # cap for all runs; early stopping may stop earlier
print("train batches:", len(train_loader))

In [ ]:
# Load Task 1's history for reference.
try:
    with open("results/metrics/mlp_history.json") as f:
        hist_task1 = json.load(f)
    with open("results/metrics/mlp_test.json") as f:
        test_task1 = json.load(f)
    print("Task 1 history loaded:", len(hist_task1["val_loss"]), "epochs,",
          "best_val_loss =", round(min(hist_task1["val_loss"]), 3),
          "| test acc =", test_task1.get("accuracy"),
          "| Part A =", test_task1.get("part_a_accuracy"))
except FileNotFoundError:
    hist_task1, test_task1 = None, None
    print("Task 1 history not found; comparison with Task 1 skipped.")

## 1. The overfitting observed in Task 1

In [ ]:
# Plot Task 1's learning curves to anchor the diagnosis.
if hist_task1 is not None:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    e = range(1, len(hist_task1["train_loss"]) + 1)
    ax[0].plot(e, hist_task1["train_loss"], label="train")
    ax[0].plot(e, hist_task1["val_loss"], label="validation")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
    ax[0].set_title("Task 1: train loss keeps dropping, val loss plateaus")
    ax[1].plot(e, hist_task1["val_acc"], color="C2")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("val acc")
    ax[1].set_title("val_acc peaks at epoch 11 (0.416) but Task 1 kept epoch 8")
    plt.tight_layout()
    Path("results/figures").mkdir(parents=True, exist_ok=True)
    plt.savefig("results/figures/mlp_diagnosis_task1.png", dpi=120)
    plt.show()

    gap_task1 = hist_task1["train_loss"][-1] - hist_task1["val_loss"][-1]
    print(f"Task 1 final train/val loss gap: {gap_task1:.3f}")
    print(f"Task 1 min val_loss = {min(hist_task1['val_loss']):.3f} at epoch {hist_task1['val_loss'].index(min(hist_task1['val_loss'])) + 1}")
    print(f"Task 1 max val_acc  = {max(hist_task1['val_acc']):.3f} at epoch {hist_task1['val_acc'].index(max(hist_task1['val_acc'])) + 1}")

## 2. Baseline (no regularisation)

Reproduce Task 1's tuned MLP from scratch on the same data, with the same epoch cap (20) as all other runs.
This is the reference for every regularised variant.

- Architecture: `make_tuned_mlp()` = 256 → 128 → 64 ReLU + He init (TP05 Ex 20).
- Optimiser: Adam lr=1e-3 (same as Task 1).

In [ ]:
torch.manual_seed(0)
model_base = make_tuned_mlp().to(device)
opt_base = torch.optim.Adam(model_base.parameters(), lr=1e-3)
hist_base, n_base = fit_regularized(
    model_base, train_loader, val_loader, loss_fn, opt_base,
    n_epochs=N_EPOCHS, device=device,
)
test_loss_base, test_acc_base = evaluate(model_base, test_loader, loss_fn, device)
print(f"baseline: {n_base} epochs | final train loss {hist_base['train_loss'][-1]:.3f} | "
      f"final val loss {hist_base['val_loss'][-1]:.3f} | test acc {test_acc_base:.3f}")

### 3a. Dropout (p = 0.2)

make_mlp(widths=[256, 128, 64], dropout=0.2) inserts nn.Dropout(0.2) after each hidden ReLU.
At each training step 20% of the activations are zeroed and the remaining 80% are scaled by
1/(1-p) = 1.25 so the mean is preserved. In eval mode, dropout is a no-op.

In [ ]:
torch.manual_seed(0)
model_drop = init_weights(make_mlp(widths=[256, 128, 64], dropout=0.2), kind="he").to(device)
opt_drop = torch.optim.Adam(model_drop.parameters(), lr=1e-3)
hist_drop, n_drop = fit_regularized(
    model_drop, train_loader, val_loader, loss_fn, opt_drop,
    n_epochs=N_EPOCHS, device=device,
)
test_loss_drop, test_acc_drop = evaluate(model_drop, test_loader, loss_fn, device)
print(f"dropout 0.2: {n_drop} epochs | final train loss {hist_drop['train_loss'][-1]:.3f} | "
      f"final val loss {hist_drop['val_loss'][-1]:.3f} | test acc {test_acc_drop:.3f}")

### 3b. Weight decay (AdamW, λ = 5e-1)

`torch.optim.AdamW` applies decoupled weight decay (Loshchilov & Hutter, 2019). This is **not** the same as
`Adam(weight_decay=...)`, which implements an L2 penalty added to the gradient, TP06 Ex 5-6 note this
distinction. With AdamW, the update is `w ← (1 - η·λ)·w - η·grad`, so large weights are shrunken at every step
regardless of the gradient magnitude.

In [ ]:
torch.manual_seed(0)
model_wd = make_tuned_mlp().to(device)
opt_wd = torch.optim.AdamW(model_wd.parameters(), lr=1e-3, weight_decay=5e-1)
hist_wd, n_wd = fit_regularized(
    model_wd, train_loader, val_loader, loss_fn, opt_wd,
    n_epochs=N_EPOCHS, device=device,
)
test_loss_wd, test_acc_wd = evaluate(model_wd, test_loader, loss_fn, device)
print(f"weight decay 5e-1: {n_wd} epochs | final train loss {hist_wd['train_loss'][-1]:.3f} | "
      f"final val loss {hist_wd['val_loss'][-1]:.3f} | test acc {test_acc_wd:.3f}")

### 3c. Early stopping (patience = 5)

Stop training when val_loss has not improved for 5 consecutive epochs (TP06 Ex 10). The weights of the best
epoch are restored at the end (`copy.deepcopy(state_dict)`), so the returned model is the checkpoint we would
ship.

In [ ]:
torch.manual_seed(0)
model_es = make_tuned_mlp().to(device)
opt_es = torch.optim.Adam(model_es.parameters(), lr=1e-3)
es = EarlyStopping(patience=5)
hist_es, n_es = fit_regularized(
    model_es, train_loader, val_loader, loss_fn, opt_es,
    n_epochs=N_EPOCHS, device=device, early_stopping=es,
)
test_loss_es, test_acc_es = evaluate(model_es, test_loader, loss_fn, device)
print(f"early stopping (patience=5): {n_es} epochs (best {es.best_epoch}) | "
      f"restored val loss {es.best_loss:.3f} | test acc {test_acc_es:.3f}")

In [ ]:
# Row builder — uses the actual parameter values from the runs above.
def summary_row(name, hist, model, sel_idx=-1):
    _, ta      = evaluate(model, train_loader, loss_fn, device)
    _, va      = evaluate(model, val_loader,  loss_fn, device)
    _, ta_test = evaluate(model, test_loader, loss_fn, device)
    return {
        "method":        name,
        "epochs_run":    len(hist["val_loss"]),
        "val_acc":       hist["val_acc"][sel_idx],
        "gap (tr-va)":   ta - va,
        "best_val_loss": min(hist["val_loss"]),
        "test_acc":      ta_test,
    }

rows = [
    summary_row("baseline",              hist_base, model_base),
    summary_row("dropout 0.2",           hist_drop, model_drop),
    summary_row("weight decay 5e-1",     hist_wd,   model_wd),
    summary_row("early stopping p=5",    hist_es,   model_es,
                sel_idx=es.best_epoch - 1),
]

import pandas as pd
comparison = pd.DataFrame(rows).set_index("method")
print(comparison.round(3))

### 3d. Comparison table and learning curves

Rows for baseline / dropout 0.2 / weight decay 5e-1 report the **last epoch** (no
selection strategy). The early-stopping row reports the **restored best epoch**
(`es.best_epoch = 6`). `gap (tr-va)` for the first three is at epoch 20, for early
stopping it is at epoch 6.

In [ ]:
# Comp 4 lines
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in [("baseline", hist_base), ("dropout 0.2", hist_drop),
                   ("weight decay 5e-1", hist_wd), ("early stopping", hist_es)]:
    e = range(1, len(hist["val_loss"]) + 1)
    ax[0].plot(e, hist["val_loss"], label=name)
    ax[1].plot(e, hist["val_acc"],  label=name)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("val loss"); ax[0].legend()
ax[0].set_title("Validation loss")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("val acc");  ax[1].legend()
ax[1].set_title("Validation accuracy")
plt.tight_layout()
Path("results/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("results/figures/mlp_diagnosis_curves.png", dpi=120)
plt.show()

## 4. Learning-rate diagnosis

Task 1 Part A used SGD lr=0.2 on a **1-layer sigmoid net** and observed oscillation
(val_acc jumping between 0.29 and 0.42). On the tuned **3-layer ReLU** network used here
SGD is not viable: lr=0.05 stays at chance level for 15 epochs (train loss 1.608 = log 5),
and lr≥0.2 diverges to NaN within the first epoch. We therefore diagnose the learning
rate with **Adam**, the same optimizer used by the baseline, so the comparison is fair.

Four learning rates are tested: 1e-4 (sweet spot), 1e-3 (baseline), 1e-2 (borderline),
1e-1 (immediate divergence — reproduces TP06 Part E "Patient 1").

In [ ]:
# SGD + momemtum 0.9 gives NaN -> Chose Adam, refer next cell
# LR_GRID = [0.05, 0.2, 1.0]
# N_EPOCHS_LR = 15
# lr_runs = {}

# for lr in LR_GRID:
#     torch.manual_seed(0)
#     m = make_tuned_mlp().to(device)
#     o = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9)
#     h, _ = fit_regularized(m, train_loader, val_loader, loss_fn, o,
#                            n_epochs=N_EPOCHS_LR, device=device)
#     _, acc = evaluate(m, test_loader, loss_fn, device)
#     lr_runs[lr] = {"hist": h, "test_acc": acc, "model": m}
#     print(f"lr={lr:>4}: final train loss {h['train_loss'][-1]:.3f} | "
#           f"final val loss {h['val_loss'][-1]:.3f} | val acc {h['val_acc'][-1]:.3f} | test acc {acc:.3f}")

In [ ]:
# LR diagnosis with Adam
# SGD+momentum on a 3-layer ReLU net either doesn't move in 15 epochs (lr=0.05)
# or diverges to NaN (lr>=0.2). Adam shows the three regimes cleanly.
LR_GRID = [1e-4, 1e-3, 1e-2, 1e-1]     # too small / baseline / too large
N_EPOCHS_LR = 20
lr_runs = {}

for lr in LR_GRID:
    torch.manual_seed(0)
    m = make_tuned_mlp().to(device)
    o = torch.optim.Adam(m.parameters(), lr=lr)
    h, _ = fit_regularized(m, train_loader, val_loader, loss_fn, o,
                           n_epochs=N_EPOCHS_LR, device=device)
    _, acc = evaluate(m, test_loader, loss_fn, device)
    lr_runs[lr] = {"hist": h, "test_acc": acc, "model": m}
    print(f"lr={lr:>6}: final train loss {h['train_loss'][-1]:.3f} | "
          f"final val loss {h['val_loss'][-1]:.3f} | "
          f"val acc {h['val_acc'][-1]:.3f} | test acc {acc:.3f}")

In [ ]:
# Plot: log-scale train loss is essential to see all four regimes on one axis.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for lr, run in lr_runs.items():
    e = range(1, len(run["hist"]["train_loss"]) + 1)
    ax[0].plot(e, run["hist"]["train_loss"], label=f"lr={lr:g}")
    ax[1].plot(e, run["hist"]["val_acc"],  label=f"lr={lr:g}")

ax[0].set_xlabel("epoch")
ax[0].set_ylabel("train loss")
ax[0].set_yscale("log")
ax[0].legend()
ax[0].set_title("Train loss vs learning rate (log scale)")

ax[1].set_xlabel("epoch")
ax[1].set_ylabel("val accuracy")
ax[1].legend()
ax[1].set_title("Val accuracy vs learning rate")

plt.tight_layout()
Path("results/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("results/figures/mlp_lr_diagnosis.png", dpi=120)
plt.show()

## 5. Which epoch should we keep?

Task 1's Conclusion left this open: `fit_with_best` selected the min-val_loss epoch, but val_acc peaked later.
Here we measure the three natural selection strategies on the same training run and compare their test
accuracy.

Method: train once for 20 epochs with the tuned MLP + Adam lr=1e-3 (baseline setting), save `state_dict` at
every epoch, then restore each candidate checkpoint and evaluate on the test set.

In [ ]:
torch.manual_seed(0)
model_sel = make_tuned_mlp().to(device)
opt_sel = torch.optim.Adam(model_sel.parameters(), lr=1e-3)

hist_sel = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
snapshots = {}       # epoch -> deepcopy(state_dict)

from museum_guide.train import train_one_epoch

for epoch in range(1, N_EPOCHS + 1):
    # Manual loop so we can snapshot at every epoch
    tr = train_one_epoch(model_sel, train_loader, loss_fn, opt_sel, device)
    _, ta = evaluate(model_sel, train_loader, loss_fn, device)
    vl, va = evaluate(model_sel, val_loader, loss_fn, device)
    hist_sel["train_loss"].append(tr)
    hist_sel["train_acc"].append(ta)
    hist_sel["val_loss"].append(vl)
    hist_sel["val_acc"].append(va)
    snapshots[epoch] = copy.deepcopy(model_sel.state_dict())

def test_acc_at(epoch_idx):
    model_sel.load_state_dict(snapshots[epoch_idx])
    _, acc = evaluate(model_sel, test_loader, loss_fn, device)
    return acc

last_ep     = len(hist_sel["val_loss"])
best_loss_ep = int(np.argmin(hist_sel["val_loss"])) + 1
best_acc_ep  = int(np.argmax(hist_sel["val_acc"]))  + 1

selection_rows = [
    {"criterion": "last epoch",  "epoch": last_ep,      "val_loss": hist_sel["val_loss"][last_ep - 1], "val_acc": hist_sel["val_acc"][last_ep - 1], "test_acc": test_acc_at(last_ep)},
    {"criterion": "min val_loss", "epoch": best_loss_ep, "val_loss": hist_sel["val_loss"][best_loss_ep - 1], "val_acc": hist_sel["val_acc"][best_loss_ep - 1], "test_acc": test_acc_at(best_loss_ep)},
    {"criterion": "max val_acc",  "epoch": best_acc_ep,  "val_loss": hist_sel["val_loss"][best_acc_ep - 1], "val_acc": hist_sel["val_acc"][best_acc_ep - 1], "test_acc": test_acc_at(best_acc_ep)},
]
selection_df = pd.DataFrame(selection_rows).set_index("criterion")
print(selection_df.round(3))

## 6. Save artefacts

Three outputs are written to disk:

- `results/metrics/mlp_diagnosis.json` — comparison table + learning-rate diagnostic + selection criterion.
- `results/metrics/mlp_epoch_selection.json` — the selection experiment alone.
- `checkpoints/mlp_regularized.pt` — the best regularised model (highest test accuracy among the four runs).

The three PNGs are already saved in the previous cells.

In [ ]:
Path("results/metrics").mkdir(parents=True, exist_ok=True)
Path("checkpoints").mkdir(parents=True, exist_ok=True)

# --- 1. Build the diagnosis dict (will include best_checkpoint after selection) ---
diagnosis = {
    "task1_reference": {
        "test_acc":   test_task1.get("accuracy")        if test_task1 else None,
        "part_a_acc": test_task1.get("part_a_accuracy") if test_task1 else None,
        "best_epoch": test_task1.get("best_epoch")      if test_task1 else None,
    },
    "regularisation": comparison.round(4).to_dict(orient="index"),
    "learning_rate": {
        str(lr): {
            "final_train_loss": run["hist"]["train_loss"][-1],
            "final_val_loss":   run["hist"]["val_loss"][-1],
            "final_val_acc":    run["hist"]["val_acc"][-1],
            "test_acc":         run["test_acc"],
        }
        for lr, run in lr_runs.items()
    },
    "epoch_selection": selection_df.round(4).to_dict(orient="index"),
}

# --- 2. Select the best model by VALIDATION accuracy (never by test accuracy:
#         choosing on test would leak the test set into model selection) ---
val_acc_by_name = {
    "baseline":       max(hist_base["val_acc"]),
    "dropout":        max(hist_drop["val_acc"]),
    "weight_decay":   max(hist_wd["val_acc"]),
    "early_stopping": hist_es["val_acc"][es.best_epoch - 1],
}
test_acc_by_name = {
    "baseline":       test_acc_base,
    "dropout":        test_acc_drop,
    "weight_decay":   test_acc_wd,
    "early_stopping": test_acc_es,
}

best_name  = max(val_acc_by_name, key=val_acc_by_name.get)
best_model = {
    "baseline":       model_base,
    "dropout":        model_drop,
    "weight_decay":   model_wd,
    "early_stopping": model_es,
}[best_name]

# Architecture-specific configs so the load recipe always matches the winning model.
best_config = {
    "baseline":       {"optimizer": "Adam",  "lr": 1e-3, "weight_decay": 0.0, "dropout": 0.0},
    "dropout":        {"optimizer": "Adam",  "lr": 1e-3, "dropout": 0.2},
    "weight_decay":   {"optimizer": "AdamW", "lr": 1e-3, "weight_decay": 5e-1, "dropout": 0.0},
    "early_stopping": {"optimizer": "Adam",  "lr": 1e-3, "weight_decay": 0.0,
                       "early_stopping_patience": 5, "dropout": 0.0},
}

# --- 3. Add best_checkpoint to the diagnosis dict (before saving) ---
diagnosis["best_checkpoint"] = {
    "name":         best_name,
    "val_acc":      val_acc_by_name[best_name],
    "test_acc":     test_acc_by_name[best_name],
    "architecture": "make_tuned_mlp (256-128-64 ReLU, He init)",
    "config":       best_config[best_name],
    "batch_size":   BATCH_SIZE,
    "seed":         0,
    "epochs":       20,
    "file":         "checkpoints/mlp_regularized.pt",
    "load_recipe":  (
        "from museum_guide.mlp import make_tuned_mlp; "
        "m = make_tuned_mlp(); "
        "m.load_state_dict(torch.load('checkpoints/mlp_regularized.pt'))"
    ),
}

# --- 4. Save JSON only now, after best_checkpoint has been added ---
with open("results/metrics/mlp_diagnosis.json", "w") as f:
    json.dump(diagnosis, f, indent=2)

with open("results/metrics/mlp_epoch_selection.json", "w") as f:
    json.dump(selection_df.round(4).to_dict(orient="index"), f, indent=2)

# --- 5. Save the checkpoint (state_dict only — the metadata lives in mlp_diagnosis.json) ---
torch.save(best_model.state_dict(), "checkpoints/mlp_regularized.pt")

# --- 6. Log the outcome ---
print(f"Best model selected by val_acc: {best_name}")
print(f"  val_acc  = {val_acc_by_name[best_name]:.3f}")
print(f"  test_acc = {test_acc_by_name[best_name]:.3f}")
print(f"  config   = {best_config[best_name]}")

## Conclusion (Task 2)

### Headline results

| method | epochs | best val_loss | test acc |
|---|---|---|---|
| baseline | 20 | 2.457 | 0.364 |
| dropout 0.2 | 20 | **1.543** | 0.356 |
| **weight decay 5e-1** | 20 | 2.149 | **0.392** |
| early stopping (p=5) | 11 | 2.457 | 0.356 |

### Q1 — Does regularisation help?

Yes, but the four methods differ:

- **Weight decay 5e-1** is the only method that improves both val_loss (2.149 vs 2.457)
  and test accuracy (0.392 vs 0.364). It is the best model of this task.
- **Dropout 0.2** gives the lowest val_loss (1.543) but its test accuracy matches
  the baseline (0.356). It under-fits relative to weight decay on this dataset.
- **Early stopping** correctly stops at epoch 11 (best epoch = 6), but because it
  uses min val_loss as the selection criterion, its test accuracy equals the
  baseline (0.356) instead of improving it.

### Q2 — Is the learning rate well chosen?

Adam lr=1e-3 (baseline) is safe but not optimal:

| lr | test acc | behaviour |
|---|---|---|
| 1e-4 | **0.384** | best generalisation |
| 1e-3 | 0.364 | safe, mild overfit |
| 1e-2 | 0.172 | learns, then collapses |
| 1e-1 | 0.144 | diverges immediately (chance level) |

`lr=1e-1` reproduces the exact failure mode from TP06 Part E "Patient 1".
SGD+momentum is not viable on this 3-layer ReLU net: lr=0.05 stays at chance
level for 15 epochs, lr≥0.2 diverges to NaN. Task 1 Part A used SGD lr=0.2
successfully on a 1-layer *sigmoid* net — the same lr is catastrophic here.

### Q3 — Which epoch should we keep?

Same training run, three selection criteria:

| criterion | epoch | val_loss | val_acc | test_acc |
|---|---|---|---|---|
| max val_acc | 19 | 2.914 | 0.408 | **0.376** |
| last epoch | 20 | 2.993 | 0.380 | 0.364 |
| min val_loss | **6** | **2.457** | 0.384 | **0.356** |

**The min-val-loss criterion used by Task 1's `fit_with_best` gives the *worst*
test accuracy here.** This answers Task 1's open question: val_loss is not the
right selection criterion. The max-val_acc vs last-epoch gap is ~3 test samples
(within noise), but min-val_loss is clearly worse than both.